# ZhongJing-TCM Benchmark · MiniMax 批量生成（Colab）

在 Colab 上端到端运行中医试题合成流水线，针对**批量生成**做了三件事：

| 能力 | 实现 |
|---|---|
| 🔌 **MiniMax API** | OpenAI 兼容端点 `https://api.minimaxi.com/v1`，模型 `MiniMax-M3` |
| ⚡ **并发生成** | `asyncio` + 信号量，多篇 passage 同时请求（`llm.max_concurrency`） |
| 💾 **断点续跑** | 每篇 passage 完成即落盘 `questions_raw.jsonl`；重连后只补缺失的 `(passage, 题型, 难度)` 三元组，不重复花 token |

完整流水线：**M1** 清洗 → **M2** 质量门控 → **M3/M4** 主题与九分类 → **M5 ⭐ 批量生成** → **M6** DTQF 过滤 → **M7** 组装打包 → **M8** 评测 → **M9** 统计。

> 💡 想先**免费**跑通整条链路？在第 4 步把 `PROVIDER` 设为 `mock`，无需任何 API key。


## 1. 安装依赖

In [ ]:
#@title 安装依赖（核心 + 可选）
# 核心依赖（流水线必需，安装很快）
!pip install -q openai anthropic tenacity diskcache "pydantic>=2" pyyaml tiktoken jieba pandas numpy scikit-learn typer tqdm chardet datasketch

# 文档解析（M1 读取 .docx / .html）
!pip install -q python-docx beautifulsoup4 lxml

# 统计 / 绘图（M2 雷达图、M7 极坐标图、M9 ANOVA/回归图）
!pip install -q statsmodels matplotlib

#@markdown 主题建模重型栈（M3 BERTopic）。体积大、首次较慢；
#@markdown 不勾选则自动使用**轻量回退**（按文章分块 + 锚词九分类，仅需 jieba）。
INSTALL_TOPIC_STACK = False  #@param {type:"boolean"}
if INSTALL_TOPIC_STACK:
    !pip install -q sentence-transformers bertopic umap-learn hdbscan gensim datasketch

#@markdown 安装中文字体，让 matplotlib 图表正确显示中文（可选）。
INSTALL_CJK_FONT = True  #@param {type:"boolean"}
if INSTALL_CJK_FONT:
    !apt-get -qq install -y fonts-noto-cjk > /dev/null 2>&1
    import matplotlib
    matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK JP", "Noto Sans CJK SC", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False

print("✅ 依赖安装完成")

## 2. 获取代码

In [ ]:
#@title 克隆仓库并切到目标分支
import os, sys

REPO_URL = "https://github.com/pariskang/ZhongJing-TCM-Benchmark.git"  #@param {type:"string"}
BRANCH   = "claude/sleepy-dijkstra-szr77x"  #@param {type:"string"}
WORKDIR  = "/content/ZhongJing-TCM-Benchmark"

if not os.path.isdir(WORKDIR):
    !git clone -q "$REPO_URL" "$WORKDIR"
%cd "$WORKDIR"
!git fetch -q origin "$BRANCH" && git checkout -q "$BRANCH" && git pull -q origin "$BRANCH"

if os.path.join(WORKDIR, "src") not in sys.path:
    sys.path.insert(0, os.path.join(WORKDIR, "src"))
print("✅ 代码就绪:", WORKDIR)

## 3.（可选）用 Google Drive 持久化断点

Colab 闲置/断线后 `/content` 会被清空。挂载 Drive 并把 `data/`、`results/` 重定向过去，
**断点文件 `questions_raw.jsonl` 就能跨重连保留**——重连后重跑第 2 步克隆代码、再跑本步建立软链，
第 9 步的生成会自动从上次进度继续。


In [ ]:
#@title 挂载 Drive 并重定向产物目录
USE_DRIVE = False  #@param {type:"boolean"}
PERSIST   = "/content/drive/MyDrive/zhongjing_tcm"  #@param {type:"string"}

if USE_DRIVE:
    import shutil, pathlib
    from google.colab import drive
    drive.mount("/content/drive")

    def _link_to_drive(sub: str):
        src = pathlib.Path(WORKDIR) / sub
        dst = pathlib.Path(PERSIST) / sub
        dst.mkdir(parents=True, exist_ok=True)
        if src.is_symlink():
            src.unlink()
        elif src.exists():
            for f in src.glob("*"):              # 把已有内容搬到 Drive（只搬一次）
                tgt = dst / f.name
                if not tgt.exists():
                    shutil.move(str(f), str(tgt))
            shutil.rmtree(src, ignore_errors=True)
        src.parent.mkdir(parents=True, exist_ok=True)
        src.symlink_to(dst, target_is_directory=True)

    for sub in ["data/raw", "data/interim", "data/final", "results"]:
        _link_to_drive(sub)
    print("✅ 断点与产物已重定向到 Drive:", PERSIST)
else:
    print("ℹ️ 未启用 Drive；断点仅存于 /content（重连会清空）。长任务建议开启。")

## 4. 配置 MiniMax / 并发 / 模型

In [ ]:
#@title 填写 API Key 与生成参数
import os, yaml
from pathlib import Path

PROVIDER        = "minimax"     #@param ["minimax", "openai", "mock"]
MINIMAX_API_KEY = ""            #@param {type:"string"}
MODEL           = "MiniMax-M3"  #@param {type:"string"}
MAX_CONCURRENCY = 8             #@param {type:"integer"}
MAX_TOKENS      = 8192          #@param {type:"integer"}
N_EVAL_RUNS     = 3             #@param {type:"integer"}

os.environ["ZHONGJING_LLM_PROVIDER"] = PROVIDER
if PROVIDER == "minimax":
    assert MINIMAX_API_KEY.strip(), "请填入 MINIMAX_API_KEY"
    os.environ["MINIMAX_API_KEY"] = MINIMAX_API_KEY.strip()
    os.environ["MINIMAX_MODEL"]   = MODEL
elif PROVIDER == "openai":
    assert MINIMAX_API_KEY.strip(), "请在 API Key 框填入 OPENAI_API_KEY"
    os.environ["OPENAI_API_KEY"]  = MINIMAX_API_KEY.strip()

# 把 provider / 并发 / 模型 / 输出上限统一写进 configs/pipeline.yaml，让所有阶段一致使用
cfg_path = Path(WORKDIR) / "configs" / "pipeline.yaml"
conf = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))
conf["llm"]["provider"] = PROVIDER
conf["llm"]["max_concurrency"] = int(MAX_CONCURRENCY)
conf["llm"]["max_tokens"] = int(MAX_TOKENS)     # 8192：避免长解析/简答被截断
if PROVIDER != "mock":
    conf["generate"]["model"]   = MODEL
    conf["quality"]["judge_model"] = MODEL
    conf["evaluate"]["models"]  = [MODEL]
conf["evaluate"]["n_runs"] = int(N_EVAL_RUNS)
cfg_path.write_text(yaml.safe_dump(conf, allow_unicode=True, sort_keys=False), encoding="utf-8")

# 让已缓存的 config / LLM client 失效，立即生效
import config as _cfg; _cfg.load_config.cache_clear()
import llm_client as _llm; _llm._DEFAULT_CLIENT = None
print(f"✅ provider={PROVIDER}  model={MODEL}  concurrency={MAX_CONCURRENCY}  max_tokens={MAX_TOKENS}  n_runs={N_EVAL_RUNS}")

## 5. 连通性自检

In [ ]:
#@title 发一条测试请求
from llm_client import call_text

if PROVIDER == "mock":
    demo = "题型: single_choice\n难度: basic\n源文本: 小柴胡汤和解少阳。"
    print("mock 输出:", call_text(demo, model="mock")[:80], "...")
else:
    try:
        out = call_text("用一句话解释中医“辨证论治”的含义。", model=MODEL, use_cache=False)
        print("✅ 模型响应:", out[:160])
    except Exception as e:
        print("❌ 调用失败，请检查 Key / 网络 / 模型名：", repr(e))

## 6. 准备输入文章

M1 支持 **`.txt` / `.md` / `.html` / `.docx`**，会**递归**读取文件夹，并自动解析杂乱文件名
（账号在 `[...]`/`【...】`、可含日期、前缀符号一律剥除，**标题取末尾正文部分**）。例如：

```
[中医书友会] - 2023-03-10 有多少大夫，正拿着“中医”的金饭碗讨饭.第22期.docx
→ 账号=中医书友会  日期=2023-03-10  标题=有多少大夫，正拿着“中医”的金饭碗讨饭.第22期
```

输入来源 `INPUT_MODE`：

- `drive_folder`：直接读取 Drive 下的文件夹（默认 `zhongjing-tcm-benchmark`）——**推荐**
- `keep`：使用仓库自带的 3 篇演示文章
- `upload` / `clear_upload`：手动上传（可先清空）


In [ ]:
#@title 选择输入来源
from pathlib import Path
import yaml
import config as _cfg

raw_dir = Path(WORKDIR) / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
EXTS = (".txt", ".md", ".html", ".htm", ".docx")

INPUT_MODE   = "drive_folder"  #@param ["drive_folder", "keep", "upload", "clear_upload"]
DRIVE_FOLDER = "/content/drive/MyDrive/zhongjing-tcm-benchmark/yichengyoudao"  #@param {type:"string"}

def _patch_raw_dir(path: str):
    """把 paths.raw_dir 指向给定目录并刷新配置缓存。"""
    cfg_path = Path(WORKDIR) / "configs" / "pipeline.yaml"
    conf = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))
    conf["paths"]["raw_dir"] = str(path)
    cfg_path.write_text(yaml.safe_dump(conf, allow_unicode=True, sort_keys=False), encoding="utf-8")
    _cfg.load_config.cache_clear()

if INPUT_MODE == "drive_folder":
    src = Path(DRIVE_FOLDER)
    if not src.exists():
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except Exception as e:
            print("⚠️ 自动挂载失败：", e)
    assert src.exists(), f"找不到 Drive 文件夹：{src}（请确认已挂载且路径正确）"
    _patch_raw_dir(src)
    raw_dir = src
    print(f"📂 输入指向 Drive 文件夹：{src}")
else:
    _patch_raw_dir(raw_dir)  # 复位到仓库内 data/raw
    if INPUT_MODE == "clear_upload":
        for f in raw_dir.glob("*"):
            if f.is_file():
                f.unlink()
    if INPUT_MODE in ("upload", "clear_upload"):
        from google.colab import files
        up = files.upload()             # 选择 .txt / .html / .docx
        for name, content in up.items():
            (raw_dir / name).write_bytes(content)

docs = [p for p in raw_dir.rglob("*") if p.is_file() and p.suffix.lower() in EXTS]
print(f"📄 待处理文档: {len(docs)}")
for f in docs[:12]:
    print("  -", f.name)

## 7. M1–M2：清洗 / 去重 / 质量门控

In [ ]:
#@title 运行 M1（清洗+去重）与 M2（三维质量评分+门控）
import m1_ingest, m2_quality
from config import load_config
load_config.cache_clear()

m1_ingest.run()
# llm_judge=True 时用所选模型给文章三维打分（专业性/科普性/实用性）；
# 想省钱可设 False，仅用启发式门控（长度/术语密度/广告比）。
USE_LLM_JUDGE = True  #@param {type:"boolean"}
articles = m2_quality.run(llm_judge=USE_LLM_JUDGE)
print("通过质量门控:", sum(a.quality_passed for a in articles), "/", len(articles))

## 8. M3–M4：主题建模 + 九分类

- 勾选了第 1 步的 BERTopic 重型栈 → 走论文同款 `sentence-BERT + HDBSCAN + c-TF-IDF`。
- 否则自动走**轻量回退**：按文章切分 passage、用 jieba 取关键词、交给 M4 锚词投票九分类。两条路都产出 `passages_labeled.jsonl`。

> ⚠️ **续跑关键**：本单元会为每个 passage 生成新的随机 `passage_id`。为保证第 9 步断点续跑有效，
> **若 `passages_labeled.jsonl` 已存在则默认跳过本步**（保持 id 稳定）。要重建请勾选 `FORCE_RELABEL`。


In [ ]:
#@title 运行 M3（主题）与 M4（九分类）
from collections import Counter
import m4_label
from config import load_config
from schemas import Article, Passage
from utils import load_jsonl_as, save_jsonl
from m3_topic import chunk_article, jieba_tok

cfg = load_config()
interim = cfg.path("paths.interim_dir")
passages_path = interim / "passages_labeled.jsonl"

FORCE_RELABEL = False  #@param {type:"boolean"}

def _build_passages_lite():
    """无 BERTopic 时的回退：每篇文章 = 一个伪主题。"""
    arts = [a for a in load_jsonl_as(interim / "articles_scored.jsonl", Article) if a.quality_passed]
    passages = []
    for tid, a in enumerate(arts):
        chunks = chunk_article(
            a,
            max_len=cfg.get("topic.max_len", 400),
            overlap=cfg.get("topic.overlap", 80),
            min_len=cfg.get("topic.min_passage_len", 50),
        )
        kw = [w for w, _ in Counter(jieba_tok(a.clean_text)).most_common(10)]
        for p in chunks:
            p.topic_id = tid
            p.topic_keywords = kw
        passages.extend(chunks)
    save_jsonl(passages, interim / "passages_topiced.jsonl")
    print(f"🪶 轻量回退: {len(passages)} passages / {len(arts)} 伪主题")

if passages_path.exists() and not FORCE_RELABEL:
    n = sum(1 for _ in passages_path.open(encoding="utf-8"))
    print(f"↩️ 已存在 passages_labeled.jsonl（{n} 行），跳过 M3/M4 以保持 passage_id 稳定。")
    print("   （要重新主题/分类，请勾选 FORCE_RELABEL。注意这会使旧断点失效。）")
else:
    try:
        import bertopic  # noqa: F401
        import m3_topic
        m3_topic.run()
        print("🧠 BERTopic 主题建模完成")
    except Exception as e:
        print("⚠️ 未用 BERTopic（", type(e).__name__, "），改用轻量回退")
        _build_passages_lite()
    m4_label.run()

labelled = [p for p in load_jsonl_as(passages_path, Passage) if p.category is not None]
print("✅ 已分类 passage:", len(labelled))

## 9. ⭐ M5：批量生成（MiniMax 并发 + 断点续跑）

核心步骤。每篇 passage 生成 `3 题型 × 3 难度 = 9` 题，**并发**请求、**完成即落盘**。

- 想限制规模先试跑：把 `LIMIT` 设为非 0（只取前 N 篇 passage）。
- **断点续跑**：若运行中断，重连后依次重跑 **第 2 步（克隆）→ 第 3 步（Drive 建链）→ 第 4 步（配置）**，
  然后**直接重跑本单元**——第 8 步会因 `passages_labeled.jsonl` 已存在而自动跳过，passage_id 保持稳定，
  `resume=True` 便只补缺失的题目。（前提：第 3 步已开启 Drive 持久化，否则 `/content` 已被清空。）


In [ ]:
#@title 生成试题（可重复运行以续跑）
import m5_generate
from config import load_config
load_config.cache_clear()

LIMIT  = 0     #@param {type:"integer"}
RESUME = True  #@param {type:"boolean"}

questions = m5_generate.run(
    limit=LIMIT or None,
    resume=RESUME,
    concurrency=MAX_CONCURRENCY,
)
print(f"✅ questions_raw.jsonl 现有 {len(questions)} 题")

# 题型 / 难度 / 类别分布速览
import pandas as pd
df = pd.DataFrame([q.model_dump(mode="json") for q in questions])
if not df.empty:
    print("\n按题型:\n", df["type"].value_counts().to_string())
    print("\n按难度:\n", df["difficulty"].value_counts().to_string())
    print("\n按类别:\n", df["category"].value_counts().to_string())

## 10. M6 DTQF 过滤 + M7 组装打包

In [ ]:
#@title 运行 M6（动态过滤）与 M7（打分块/分层/数据卡）
import m6_dtqf, m7_assemble
from config import load_config
load_config.cache_clear()

m6_dtqf.run()
final_qs = m7_assemble.run()
print("✅ 通过 DTQF 并组装:", len(final_qs), "题")

# 展示数据卡
from utils import load_jsonl_as
from schemas import Question
import pandas as pd
card = m7_assemble.dataset_card(final_qs)
print("\n=== 数据集概览 (dataset_card) ===")
print(card.to_string() if hasattr(card, "to_string") else card)
print("\n产物：")
!ls -lh "$WORKDIR"/data/final

## 11.（可选）M8 评测 + M9 统计

用所选模型对**诊断子集**做 STAGER 零样本评测（含简答题语义判分），再跑 ANOVA / 回归 / DP 分段。
注意：这会再次消耗 API 额度。


In [ ]:
#@title 评测与统计
RUN_EVAL = True  #@param {type:"boolean"}
if RUN_EVAL:
    import m8_evaluate, m9_stats
    from config import load_config
    load_config.cache_clear()

    metrics = m8_evaluate.run(MODEL if PROVIDER != "mock" else "mock")
    print("📊 评测指标:", metrics)

    stats = m9_stats.run()
    print("📈 统计完成:", list(stats.keys()) if stats else "（无评测记录）")
    !ls -lh "$WORKDIR"/results 2>/dev/null
else:
    print("已跳过评测/统计。")

## 12. 导出结果

In [ ]:
#@title 打包并下载 data/final + results
import os
os.chdir(WORKDIR)
!zip -qr /content/zhongjing_outputs.zip data/final results -x "*.gitkeep"
print("✅ 已打包 -> /content/zhongjing_outputs.zip")
try:
    from google.colab import files
    files.download("/content/zhongjing_outputs.zip")
except Exception as e:
    print("（非 Colab 环境，跳过自动下载）", e)

---
### 小贴士

- **续跑**：开启第 3 步 Drive 持久化后，断线只需重跑「克隆 → 建链 → 配置 → 第 9 步」，生成自动接力。
- **并发上限**：`MAX_CONCURRENCY` 视账号 QPS 调整；遇到限流报错就调小。客户端已内置指数退避重试。
- **省钱**：M2 的 `USE_LLM_JUDGE=False` 可跳过文章打分；M5 的 `LIMIT` 可先小规模验证再全量。
- **缓存**：相同 prompt 命中本地 `diskcache`，重跑不重复计费。
- **换模型**：把第 4 步 `MODEL` 改成任意 MiniMax 模型名即可（如 `MiniMax-Text-01`）。
